# Raw-source audit and face-video synchronization

This notebook checks the optical pulse candidate that links the face video to the 28,801 neural samples. It treats the pulse correspondence as strong empirical evidence, while reserving formal hardware interpretation until acquisition metadata are available.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from yumeow_plot import get_fig

ROOT = Path('..').resolve()
SYNC_DIR = ROOT / 'logs/scripts/recover_video_sync/initial'
VIDEO_DIR = ROOT / 'logs/scripts/extract_align_video/initial'
OUT = ROOT / 'figures/261027原始数据与视频同步'
OUT.mkdir(parents=True, exist_ok=True)
sync = json.loads((SYNC_DIR / 'summary.json').read_text())
pulses = pd.read_csv(SYNC_DIR / 'candidate_frame_timestamps.csv')
state_video = pd.read_csv(VIDEO_DIR / 'aligned_state_video_summary.csv')
sync

{'threshold_green_excess': -12.25,
 'total_video_frames_decoded': 251285,
 'all_rising_edges': 28801,
 'all_falling_edges': 28801,
 'longest_pulse_run_count': 28801,
 'expected_neural_frames': 28801,
 'frame_count_difference': 0,
 'first_pulse_video_seconds': 13.6,
 'last_pulse_video_seconds': 6959.566666666667,
 'pulse_run_duration_seconds': 6945.966666666666,
 'linear_clock_intercept_seconds': 13.582669957421219,
 'linear_seconds_per_neural_frame': 0.24117809564198903,
 'linear_sampling_hz': 4.146313525439001,
 'timing_residual_std_seconds': 0.025981515580041165,
 'timing_residual_max_absolute_seconds': 0.08173971259111568,
 'median_interpulse_seconds': 0.23333333333357587,
 'unique_interpulse_video_frames': [7, 8, 9],
 'pulse_width_video_frames_median': 3.0,
 'pulse_width_video_frames_range': [2.0, 4.0],
 'pulse_width_seconds_median': 0.1,
 'pulse_width_seconds_p10_p90': [0.06666666666666667, 0.1],
 'interpretation': 'Candidate only: correspondence is strong if the longest run has 2

In [2]:
fig, axes = get_fig(RN=1, CN=2, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(axes)
ax = next(ax_loader)
thin = np.arange(0, len(pulses), 20)
ax.plot(pulses.candidate_neural_frame.iloc[thin], 1e3 * pulses.residual_seconds.iloc[thin], lw=.7, color='#3366aa')
ax.axhline(0, color='0.3', lw=.8)
ax.set(xlabel='Neural frame', ylabel='Pulse timing residual (ms)', title='Linear clock residual')
ax = next(ax_loader)
bins = np.arange(1.5, 5.6, 1)
ax.hist(pulses.pulse_width_video_frames, bins=bins, color='#dd7755', edgecolor='white')
ax.set(xlabel='Pulse width (video frames)', ylabel='Count', title='Optical pulse width')
fig.savefig(OUT / 'sync_timing_and_pulse_width.png')
fig.savefig(OUT / 'sync_timing_and_pulse_width.svg')
fig.savefig(OUT / 'sync_timing_and_pulse_width.pdf')
plt.show()

In [3]:
fig, axes = get_fig(RN=1, CN=1, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(axes)
ax = next(ax_loader)
colors = {'Awake': '#cc5544', 'NREM': '#4477aa', 'REM': '#aa55aa'}
x = np.arange(len(state_video))
means = state_video.face_motion_mean.to_numpy()
errs = state_video.face_motion_std.to_numpy()
ax.bar(x, means, yerr=errs, color=[colors[s] for s in state_video.state], alpha=.85, capsize=3)
ax.set_xticks(x, state_video.state)
ax.set(ylabel='Face-motion score', title='State-conditioned face motion')
fig.savefig(OUT / 'state_face_motion.png')
fig.savefig(OUT / 'state_face_motion.svg')
fig.savefig(OUT / 'state_face_motion.pdf')
plt.show()

## Interpretation

The longest uninterrupted green-light pulse train contains exactly 28,801 rising edges, matching the neural frame count. A linear clock fit gives 4.1463135 Hz; residual scatter is about 26 ms, comparable to video-frame quantization. Pulse widths are mostly 2–4 video frames (median 0.10 s). This supports a one-pulse-per-neural-frame synchronization candidate. The pulse width is only an optical proxy for the acquisition/integration window, not documented exposure time.

The aligned face-motion summary is a useful state covariate and quality-control signal. It should not be treated as an independent validation of the offset because state and motion vary slowly and behavior-only correlation has a broad, weak optimum.